# Bitcoin Wallet Strategy: Cold vs Hot Risk-Cost Analyzer


In [ ]:
# Install dependencies
!pip install -q requests pandas matplotlib numpy


# Bitcoin Wallet Strategy: Cold vs Hot Risk-Cost Analyzer

This notebook models the security, cost, and operational trade-offs between cold wallets (hardware, ~$50–200, ~0.1% breach risk) and hot wallets (exchanges/software, free–low cost, ~5% annual breach risk) based on portfolio size and trading frequency.

**Source**: Analysis informed by cold wallet vs hot wallet security comparison.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.interpolate import interp1d

# Define wallet parameters
cold_wallet_cost = 100  # USD, one-time hardware cost
hot_wallet_cost = 0     # Assume free exchange/software wallet

cold_breach_risk_annual = 0.001  # 0.1% annual breach risk
hot_breach_risk_annual = 0.05     # 5% annual breach risk (exchange hack, phishing, etc.)

trading_friction_cost_hot = 0.002  # 0.2% per transaction (faster access)
trading_friction_cost_cold = 0.01   # 1% equivalent friction (slower, manual verification)

print("=" * 60)
print("WALLET RISK & COST PARAMETERS")
print("=" * 60)
print(f"Cold Wallet: ${cold_wallet_cost} one-time, {cold_breach_risk_annual*100:.2f}% breach risk/year")
print(f"Hot Wallet: ${hot_wallet_cost} cost, {hot_breach_risk_annual*100:.1f}% breach risk/year")
print(f"Trading friction: Hot {trading_friction_cost_hot*100:.1f}%, Cold {trading_friction_cost_cold*100:.1f}%")

In [ ]:
# Model cumulative loss by allocation and time horizon

def calculate_risk_adjusted_loss(portfolio_usd, cold_pct, years=1):
    """
    Calculate expected loss from security breach by allocation.
    
    Args:
        portfolio_usd: Total portfolio in USD
        cold_pct: Percentage held in cold wallet (0-1)
        years: Time horizon
    
    Returns:
        Expected loss in USD
    """
    hot_pct = 1 - cold_pct
    
    cold_value = portfolio_usd * cold_pct
    hot_value = portfolio_usd * hot_pct
    
    # Expected loss from breach (simplified: single-year model)
    cold_loss = cold_value * cold_breach_risk_annual * years
    hot_loss = hot_value * hot_breach_risk_annual * years
    
    total_loss = cold_loss + hot_loss
    return total_loss

def calculate_trading_friction_cost(portfolio_usd, cold_pct, annual_trades=12):
    """
    Calculate transaction friction cost.
    
    Args:
        portfolio_usd: Total portfolio
        cold_pct: Percentage in cold wallet
        annual_trades: Number of trades/rebalances per year
    
    Returns:
        Annual friction cost in USD
    """
    hot_pct = 1 - cold_pct
    hot_friction = portfolio_usd * hot_pct * trading_friction_cost_hot * annual_trades
    cold_friction = portfolio_usd * cold_pct * trading_friction_cost_cold * annual_trades
    return hot_friction + cold_friction

# Test
portfolio = 50000  # $50k portfolio
print(f"\nTest (${portfolio:,} portfolio, 70% cold, 12 trades/year):")
print(f"  Risk-adjusted loss: ${calculate_risk_adjusted_loss(portfolio, 0.7, years=1):,.2f}")
print(f"  Trading friction: ${calculate_trading_friction_cost(portfolio, 0.7, annual_trades=12):,.2f}")

In [ ]:
# Generate decision matrix: optimal allocation by portfolio size & trading frequency

portfolio_sizes = np.array([1000, 5000, 10000, 50000, 100000, 500000])
trading_frequencies = np.array([0, 4, 12, 52])  # annually: hold, quarterly, monthly, weekly

# For each combo, find cold_pct that minimizes total cost
results = []

for portfolio in portfolio_sizes:
    for trades in trading_frequencies:
        best_cold_pct = 0
        min_cost = float('inf')
        
        for cold_pct in np.linspace(0, 1, 11):
            hw_cost = cold_wallet_cost if cold_pct > 0 else 0  # Only pay if using cold wallet
            risk_loss = calculate_risk_adjusted_loss(portfolio, cold_pct, years=1)
            friction = calculate_trading_friction_cost(portfolio, cold_pct, annual_trades=trades)
            total_cost = hw_cost + risk_loss + friction
            
            if total_cost < min_cost:
                min_cost = total_cost
                best_cold_pct = cold_pct
        
        results.append({
            'Portfolio': portfolio,
            'Annual Trades': trades,
            'Optimal Cold %': best_cold_pct * 100,
            'Total Annual Cost': min_cost
        })

df_results = pd.DataFrame(results)
print("\n" + "="*70)
print("OPTIMAL ALLOCATION BY PORTFOLIO SIZE & TRADING FREQUENCY")
print("="*70)
for portfolio in portfolio_sizes:
    print(f"\n${portfolio:,} Portfolio:")
    subset = df_results[df_results['Portfolio'] == portfolio]
    for _, row in subset.iterrows():
        print(f"  {int(row['Annual Trades']):2d} trades/year → {row['Optimal Cold %']:5.0f}% cold (cost: ${row['Total Annual Cost']:8.2f})")

In [ ]:
# Visualization: cost breakdown across allocation strategies

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Cold vs Hot Wallet: Risk-Cost Analysis', fontsize=14, fontweight='bold')

# 1. Risk-adjusted loss by allocation (fixed $50k portfolio)
ax = axes[0, 0]
portfolio_fixed = 50000
cold_pcts = np.linspace(0, 1, 21)
risk_losses = [calculate_risk_adjusted_loss(portfolio_fixed, cp, years=1) for cp in cold_pcts]
ax.plot(cold_pcts * 100, risk_losses, marker='o', linewidth=2, color='#e74c3c')
ax.set_xlabel('Cold Wallet Allocation (%)', fontsize=10)
ax.set_ylabel('Annual Risk-Adjusted Loss (USD)', fontsize=10)
ax.set_title(f'Security Risk by Allocation (${portfolio_fixed:,})', fontsize=11)
ax.grid(True, alpha=0.3)

# 2. Trading friction cost by allocation (monthly trading)
ax = axes[0, 1]
friction_costs = [calculate_trading_friction_cost(portfolio_fixed, cp, annual_trades=12) for cp in cold_pcts]
ax.plot(cold_pcts * 100, friction_costs, marker='s', linewidth=2, color='#3498db')
ax.set_xlabel('Cold Wallet Allocation (%)', fontsize=10)
ax.set_ylabel('Annual Friction Cost (USD)', fontsize=10)
ax.set_title('Trading Friction by Allocation (12 trades/year)', fontsize=11)
ax.grid(True, alpha=0.3)

# 3. Total cost by allocation (hardware + risk + friction)
ax = axes[1, 0]
total_costs = []
for cp in cold_pcts:
    hw = cold_wallet_cost if cp > 0 else 0
    risk = calculate_risk_adjusted_loss(portfolio_fixed, cp, years=1)
    friction = calculate_trading_friction_cost(portfolio_fixed, cp, annual_trades=12)
    total_costs.append(hw + risk + friction)

ax.plot(cold_pcts * 100, total_costs, marker='D', linewidth=2.5, color='#27ae60')
optimal_idx = np.argmin(total_costs)
ax.axvline(cold_pcts[optimal_idx] * 100, color='red', linestyle='--', alpha=0.7, label=f'Optimal: {cold_pcts[optimal_idx]*100:.0f}%')
ax.scatter([cold_pcts[optimal_idx] * 100], [total_costs[optimal_idx]], color='red', s=100, zorder=5)
ax.set_xlabel('Cold Wallet Allocation (%)', fontsize=10)
ax.set_ylabel('Total Annual Cost (USD)', fontsize=10)
ax.set_title('Total Cost: Hardware + Risk + Friction', fontsize=11)
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Recommendation heatmap: cold % by portfolio size & trading frequency
ax = axes[1, 1]
portfolios = np.array([1000, 10000, 50000, 100000, 500000])
trades_per_year = np.array([0, 4, 12, 52])
heatmap_data = np.zeros((len(trades_per_year), len(portfolios)))

for i, trades in enumerate(trades_per_year):
    for j, pf in enumerate(portfolios):
        best_pct = 0
        min_cost = float('inf')
        for cp in np.linspace(0, 1, 21):
            hw = cold_wallet_cost if cp > 0 else 0
            risk = calculate_risk_adjusted_loss(pf, cp, years=1)
            friction = calculate_trading_friction_cost(pf, cp, annual_trades=trades)
            total = hw + risk + friction
            if total < min_cost:
                min_cost = total
                best_pct = cp * 100
        heatmap_data[i, j] = best_pct

im = ax.imshow(heatmap_data, cmap='RdYlGn', aspect='auto', vmin=0, vmax=100)
ax.set_xticks(range(len(portfolios)))
ax.set_yticks(range(len(trades_per_year)))
ax.set_xticklabels([f'${p/1000:.0f}k' for p in portfolios], fontsize=9)
ax.set_yticklabels([f'{t} trades/yr' for t in trades_per_year], fontsize=9)
ax.set_xlabel('Portfolio Size', fontsize=10)
ax.set_ylabel('Trading Frequency', fontsize=10)
ax.set_title('Optimal Cold Wallet Allocation (%)', fontsize=11)

for i in range(len(trades_per_year)):
    for j in range(len(portfolios)):
        text = ax.text(j, i, f'{heatmap_data[i, j]:.0f}%',
                      ha="center", va="center", color="black", fontsize=9, fontweight='bold')

plt.colorbar(im, ax=ax, label='Cold % Recommended')
plt.tight_layout()
plt.savefig('wallet_analysis.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✓ Visualization saved.")

In [ ]:
# Decision helper: given your profile, what should you do?

print("\n" + "="*70)
print("WALLET ALLOCATION RECOMMENDATION TOOL")
print("="*70)

def recommend_allocation(portfolio_usd, annual_transactions, technical_comfort='medium'):
    """
    Simple recommendation based on profile.
    
    Args:
        portfolio_usd: Total holdings
        annual_transactions: Number of trades/rebalances per year
        technical_comfort: 'low', 'medium', 'high'
    """
    cold_pct = 0.5  # Default
    
    # Adjust by portfolio size
    if portfolio_usd < 5000:
        cold_pct = 0.2  # Small portfolio: hot wallet preferred (friction > security gain)
    elif portfolio_usd < 50000:
        cold_pct = 0.5  # Medium: balanced
    else:
        cold_pct = 0.7  # Large: cold preferred (security gain > friction)
    
    # Adjust by trading frequency
    if annual_transactions > 24:
        cold_pct *= 0.7  # Frequent trader: reduce cold %
    elif annual_transactions < 4:
        cold_pct = max(cold_pct, 0.8)  # Hodler: maximize cold %
    
    # Adjust by technical comfort
    if technical_comfort == 'low':
        cold_pct *= 0.5
    elif technical_comfort == 'high':
        cold_pct = min(cold_pct * 1.2, 1.0)
    
    return cold_pct

# Examples
scenarios = [
    {'name': 'New trader, $5k, monthly buys', 'portfolio': 5000, 'trades': 12, 'comfort': 'low'},
    {'name': 'Investor, $100k, quarterly rebalance', 'portfolio': 100000, 'trades': 4, 'comfort': 'medium'},
    {'name': 'Hodler, $250k, minimal trading', 'portfolio': 250000, 'trades': 1, 'comfort': 'high'},
    {'name': 'Day trader, $50k, weekly active', 'portfolio': 50000, 'trades': 52, 'comfort': 'medium'},
]

for s in scenarios:
    pct = recommend_allocation(s['portfolio'], s['trades'], s['comfort'])
    cold_amt = s['portfolio'] * pct
    hot_amt = s['portfolio'] * (1 - pct)
    print(f"\n{s['name']}")
    print(f"  → {pct*100:.0f}% cold (${cold_amt:,.0f}), {(1-pct)*100:.0f}% hot (${hot_amt:,.0f})")

---

## Reference

[https://protraderdaily.com](https://protraderdaily.com/crypto/bitcoin-cold-wallet-vs-hot-wallet)
